# 04 · Obesidad: modelado

Proyecto del curso Machine Learning (650064), Universidad de Lima, ciclo 2026-2.  
**Objetivo del notebook:** entrenar y comparar los modelos de referencia y los métodos nuevos (KNN y MLP) en los dos escenarios; el escenario principal es E2 (sin Height ni Weight).  
**Decisiones:** `documentacion/diseno_experimental.md` (las secciones del diseño se citan como §).

## Autores por sección

| Sección | Autor |
|---|---|
| Todas las secciones de este notebook | Marcelo Cueto |
| Funciones de `utilidades.py` | Marcelo Cueto |

## 0. Configuración

**Qué hacemos:** preparamos el entorno y cargamos las librerías.

**Por qué:** la celda 0 hace que el mismo notebook funcione en VS Code y en Colab, con las versiones fijadas en `requirements.txt`.

In [ ]:
# Celda 0. Configuración del entorno: funciona igual en VS Code y en Colab.
import os
import sys

EN_COLAB = "google.colab" in sys.modules
USUARIO_GITHUB = "MCuetoV"
REPOSITORIO = "proyecto-ml-2026-2"

if EN_COLAB:
    # En Colab: clonar el repositorio (o actualizarlo) con el token guardado en Secrets
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not os.path.exists(f"/content/{REPOSITORIO}"):
        !git clone -q https://{token}@github.com/{USUARIO_GITHUB}/{REPOSITORIO}.git /content/{REPOSITORIO}
    os.chdir(f"/content/{REPOSITORIO}")
    !git pull -q
    !pip install -q -r requirements.txt
else:
    # En VS Code: si el notebook se abrió desde notebooks/, subir a la raíz del repositorio
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("Carpeta de trabajo:", os.getcwd())

In [ ]:
# Librerías y funciones comunes del proyecto
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import utilidades

pd.set_option("display.max_columns", 50)
print("Semilla del proyecto:", utilidades.SEMILLA)

## 1. Carga, escenarios y partición

**Qué hacemos:** cargamos los datos, recuperamos la partición del notebook de exploración y definimos las variables de cada escenario.

**Por qué:** todos los modelos se comparan con las mismas filas y las mismas variables (§7).

In [ ]:
# Datos, partición única (§7) y variables de cada escenario (§1)
datos = utilidades.cargar_obesidad()
train, test = utilidades.dividir_train_test(datos, "obesidad")
y_train, y_test = train[utilidades.OBJETIVO_OBESIDAD], test[utilidades.OBJETIVO_OBESIDAD]
grupos_train = train["grupo"]   # para la CV agrupada (§7)
ESCENARIOS = {esc: utilidades.columnas_por_escenario("obesidad", esc) for esc in ["E1", "E2"]}
for esc, columnas in ESCENARIOS.items():
    print(esc, "→", len(columnas), "variables")
print("Train:", len(train), "· Test:", len(test))
filas_resultados = []   # registro de resultados de este notebook (§11.2)

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 2. Pipeline de preprocesamiento

**Qué hacemos:** revisamos cuántas columnas produce la codificación en cada escenario.

**Por qué:** la codificación y el escalado van dentro del `Pipeline`, así que en la CV se aprenden en cada fold y nunca ven la validación ni el test.

In [ ]:
# Preprocesador de cada escenario (§4.2): se ajusta solo con el train
for esc, columnas in ESCENARIOS.items():
    preprocesador = utilidades.crear_preprocesador("obesidad", esc)
    matriz = preprocesador.fit_transform(train[columnas])
    print(esc, "→", matriz.shape[1], "columnas después de codificar")

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 3. Modelos de referencia

**Qué hacemos:** entrenamos el piso trivial y el método de los autores.

**Por qué:** el Dummy es el piso; el árbol reproduce el J48 de los autores en nuestras mismas condiciones. CART no es C4.5 y se declara (§2.2).

In [ ]:
# Fase 3: DummyClassifier(strategy="most_frequent") y árbol de decisión (criterion="entropy",
# aproximación a J48) con GridSearchCV. CV: utilidades.particiones_agrupadas(y_train, grupos_train, SEMILLA)

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 4. Ajuste con Optuna: KNN y MLP

**Qué hacemos:** buscamos los hiperparámetros de KNN (50 *trials*) y del MLP (100 *trials*) con TPE, en cada escenario.

**Por qué:** cumple las estrategias (a) y (b) de la consigna; el objetivo es el macro-F1 medio de una CV de 5 folds sobre el train (§8).

In [ ]:
# Fase 3: función objetivo por modelo (espacios de §8) que devuelve
# utilidades.evaluar_cv(pipeline, X, y, grupos=grupos_train).mean(); optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEMILLA))

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 5. GridSearchCV frente a Optuna

**Qué hacemos:** comparamos la malla completa de KNN (200 combinaciones) con 50 *trials* de Optuna.

**Por qué:** GridSearchCV es la herramienta del curso; la comparación mide calidad, número de evaluaciones y tiempo (§8).

In [ ]:
# Fase 3: tabla_05_grid_vs_optuna_obesidad (mejor macro-F1, evaluaciones, segundos)

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 6. Balanceo de clases

**No aplica en obesidad.** Las siete clases ya están balanceadas porque los autores generaron el 77% de las filas con SMOTE; volver a sobremuestrear multiplicaría las filas sintéticas (§5).

## 7. Selección de variables

**Qué hacemos:** agregamos un selector L1 (Lasso logístico) al `Pipeline` y lo comparamos con usar todas las variables.

**Por qué:** la consigna pide selección de *features*; dentro del `Pipeline` se aprende en cada fold y evita el sesgo de seleccionar con todos los datos (§6).

In [ ]:
# Fase 3: SelectFromModel(LogisticRegression(penalty="l1", solver="saga", C=1.0)) entre el preprocesador y el modelo; CV 5×3

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 8. Comparativa por validación cruzada

**Qué hacemos:** evaluamos todas las configuraciones finales con CV 5×3 y otra semilla.

**Por qué:** la comparativa reporta media ± desviación y una prueba de Wilcoxon contra la referencia de los autores (§10).

In [ ]:
# Fase 3: utilidades.evaluar_cv(..., repeticiones=3, semilla=utilidades.SEMILLA_COMPARATIVA)
# → tabla_03_comparativa_cv_obesidad y tabla_06_ablaciones_obesidad

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 8.1 Sensibilidad del protocolo

**Qué hacemos:** repetimos la CV con el protocolo ingenuo (sin redondear y sin agrupar).

**Por qué:** cuantifica cuánto inflan las métricas la huella de SMOTE y las copias (§7.6).

In [ ]:
# Fase 3: tabla_07_sensibilidad_protocolo_obesidad

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 9. Evaluación final en el test

**Qué hacemos:** reentrenamos cada configuración final con todo el train y la evaluamos una sola vez.

**Por qué:** el test de reserva da la estimación sin sesgo; no se cambia nada después de verlo (§7.5).

In [ ]:
# Fase 3: utilidades.metricas_test(modelo, X_test, y_test, "obesidad") → tabla_04_metricas_test_obesidad
# Además: métricas restringidas a test["probablemente_real"] (§7.6)

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 10. Interpretación del mejor modelo

**Qué hacemos:** analizamos la matriz de confusión, la importancia por permutación y la curva de aprendizaje.

**Por qué:** explican qué aprendió el modelo y si sufre de sesgo o de varianza (§10). Revisar el peso de Gender (artefacto del sobremuestreo).

In [ ]:
# Fase 3: figura_04_confusion_obesidad, figura_05_importancia_obesidad, figura_06_curva_aprendizaje_obesidad

**Qué observamos:** _(se completa después de ejecutar la celda anterior)._

## 11. Registro de resultados

**Qué hacemos:** guardamos todas las filas registradas en el archivo de resultados de este notebook.

**Por qué:** el reporte toma sus cifras solo de `resultados/` (§11.2).

In [ ]:
# Fase 3: utilidades.guardar_tabla(pd.DataFrame(filas_resultados), "resultados_obesidad")

## Guardar resultados en GitHub (solo corrida oficial en Colab)

**Qué hacemos:** subimos a `main` los archivos de `resultados/` que genera este notebook.

**Por qué:** las cifras del reporte deben salir de la misma corrida que deja las salidas visibles en el notebook (§12, regla 4). En VS Code esta celda no hace nada; Marcelo hace el *commit* desde VS Code. Después de esta celda, guardar el notebook con *File → Save a copy in GitHub*.

In [ ]:
# Celda final. Cambiar a True solo en la corrida oficial, después de "Reiniciar y ejecutar todo".
import glob

GUARDAR_EN_GITHUB = False
PATRONES_DE_ESTE_NOTEBOOK = ["resultados/tablas/tabla_0[2-7]_*obesidad.csv", "resultados/tablas/resultados_obesidad.csv", "resultados/figuras/figura_0[4-8]_*obesidad.png"]
if EN_COLAB and GUARDAR_EN_GITHUB:
    archivos = " ".join(f for patron in PATRONES_DE_ESTE_NOTEBOOK for f in sorted(glob.glob(patron)))
    nombre, correo = userdata.get("GITHUB_NOMBRE"), userdata.get("GITHUB_CORREO")
    !git config user.name "{nombre}"
    !git config user.email "{correo}"
    !git add {archivos}
    !git commit -q -m "04-obesidad: actualiza resultados de la corrida oficial"
    !git pull -q --rebase
    !git push -q
    print("Archivos subidos:", archivos)